In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import random
import time
import scimap as sm
import anndata as ad
from functools import partial
from helperFunctions import *
from smallestEnclosingCircle import make_circle
from sklearn.mixture import GaussianMixture
from ecd_helperFunctions import *

Running SCIMAP  2.1.1


### Topographical Correlation Map Feature Based Cox-PH Model

**Step 1: Data Preprocessing**
- Load data into correct format for TCM
- Divide data into ROIs and save as .hd5 files<br>

**Step 2: Generate Complete Spatial Randomness Datasets**
- Generate datasets with equal number of cells as each ROI
- Create "paired" CSR datasets for each ROI<br>

**Step 3: Calculate Topographical Correlation Maps**
- Calculate TCM for both real and CSR datasets<br>

**Step 4: Compare Positive and Negative TCM Distributions**
- For positive TCM values, use K-S test to compare real and CSR datasets
- For negative TCM values, use K-S test to compare real and CSR datasets<br>

**Step 5: Rank ROIs based on K-S Score**
- Use rank to select the top 10 ROIs with the most positive correlation and top 10 ROIs with the most negative correlation between cell markers<br>

**Step 6: Calculate the distance between the centroids of the top 10 ROIs**
- Calculate the pairwise distance between the of the top 10 postive and negative ROIs
- This will result pos_TCM_dist_12, pos_TCM_dist_23, pos_TCM_dist_34, etc. for positive TCM values and neg_TCM_dist_12, neg_TCM_dist_23, neg_TCM_dist_34, etc. for negative TCM values<br>

**Step 7: Calculate the Cox-PH model with L2 regularization**
- The model will have 29 covariates added per marker pair that is assessed in the TCM

### **STEP ONE**: Data Preprocessing

In [2]:
# Get all .csv files in one list
data_dir = '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/'
data_parent_dir = os.listdir(data_dir)

file_paths = []
for i in range(len(data_parent_dir)):
    files = os.listdir(data_dir + data_parent_dir[i])
    csv_files = [file for file in files if file.endswith('.csv')]
    for file in csv_files:
        file_path = os.path.join(data_dir, data_parent_dir[i], file)
        file_paths.append(file_path)

print(file_paths)

['/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC07/P37_S35-CRC07.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC35/P37_S78-CRC35.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC14/P37_S46-CRC14.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC26/P37_S63-CRC26.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC21/P37_S58-CRC21.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC19/P37_S51-CRC19.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC13/P37_S45-CRC13.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC32/P37_S75-CRC32.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC38/P37_S81-CRC38.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC33_02/P37_S76_02-CRC33_02.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC04/P37_S32-C

In [ ]:
# Define markers of interest (reduce the file size needed to save)
markers = ['CD45', 'CD4', 'SMA', 'PD-L1', 'Pan-CK']
grid_directory = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'

##### Only run the below step if you want to generate NEW grid files for the TCM

In [ ]:
# Apply GMM to each file and save the grid
for f in file_paths:
    df = apply_gmm([f], markers)
    grid_file_name = grid_directory + f.split('/')[-1].split('.')[0] + '_gmm.h5'
    create_tile_rois(df, tile_size=5000, save=True, save_hdf5=grid_file_name)

### **STEP TWO/THREE/FOUR**: Generate Complete Spatial Randomness Datasets, Calculate TCMs, and Calculate K-S Scores
This is very computationally intensive. Do not run this in a notebook on an entire dataset. Instead, run this in a script on a cluster with multithreading. Multithreading functionality can be achieved with the helper function `ecd_helperFunctions.multithread_compare_tcm()`.

In [ ]:
cwd = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids'
short_grid_files = os.listdir(cwd)
grid_files = [os.path.join(cwd, f) for f in short_grid_files]
markers = ['CD45_status', 'PD-L1_status']
keep_cols = ['X_centroid', 'Y_centroid', 'CellID']
labels  = {1: 'Lymphocytes',
            2: 'PD-L1'}
rename_cols_dict = {'X_centroid': 'x', 
                    'Y_centroid': 'y'}
save_tcm_plot_path = '/michorlab/ecdyer/multiplex_spatial/figures/tcm_plots/'
save_csr_plot_path = '/michorlab/ecdyer/multiplex_spatial/figures/csr_tcm_plots/'
save_ks_results_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'
save_tcm_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'
save_csr_tcm_path = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_pdl1/'

for grid_file in grid_files:
    compare_tcm(grid_file, 
        markers,
        keep_cols,
        labels,
        visualiseStages=False,
        #save_tcm_plot_path=save_tcm_plot_path,
        #save_csr_plot_path=save_csr_plot_path,
        #save_tcm_path=save_tcm_path,
        #save_csr_tcm_path=save_csr_tcm_path,
        save_ks_results_path=save_ks_results_path,
        rename_cols_dict=rename_cols_dict,
        plot_point_cloud=False)

### **STEP FIVE/SIX**: Rank ROIs and Calculate Pairwise Distances

In [23]:
def rank_and_calculate_distance(ks_results_dir, 
                                grid_files,
                                compare_n = 20,
                                ks_stat='ks_stat',
                                save_dir=None):
    """
    Ranks and calculates distance for the top results based on the KS statistic.

    Parameters:
    - ks_results_dir (str): The directory path where the KS results are stored.
    - grid_files (str): The directory path where the grid files are stored.
    - compare_n (int, optional): The number of top results to consider. Default is 20.
    - ks_stat (str, optional): The column name of the KS statistic in the results. Default is 'ks_stat'.
    - save_dir (str, optional): The directory path to save the top results. Default is None.

    Returns:
    - ks_rank_grid_dict (dict): A dictionary containing the top results for each sample.

    """
    grid_dict = {}
    ks_rank_grid_dict = {}
    grid_file_list = os.listdir(grid_files)
    for g in grid_file_list:
        grid_sample_name = get_crc_sample_label(g)
        grid_dict[grid_sample_name] = grid_files + '/' + g
    for f in os.listdir(ks_results_dir):
        if f.endswith('.csv'):
            ks_sample_name = f[:5]
            ks_results = pd.read_csv(os.path.join(ks_results_dir, f))
            top_ks_results = ks_results.nlargest(compare_n, ks_stat)
            ks_result_grid_file = grid_dict[ks_sample_name]
            ks_result_grid = load_tile_rois(ks_result_grid_file)
            # Initialize new columns in top_ks_results
            top_ks_results['X_centroid'] = None
            top_ks_results['Y_centroid'] = None

            # Convert keys from tuples to strings
            ks_result_grid = {str(k): v for k, v in ks_result_grid.items()}
            
            # Iterate over each row in top_ks_results
            for index, row in top_ks_results.iterrows():
                grid_location = row['grid_location']

                # Locate the corresponding DataFrame in ks_result_grid
                if grid_location in ks_result_grid.keys():
                    grid_data = ks_result_grid[grid_location]
                    
                    # Calculate the centroid values
                    x_centroid = grid_data['X_centroid'].mean()
                    y_centroid = grid_data['Y_centroid'].mean()
                    
                    # Add the centroid values to top_ks_results
                    top_ks_results.at[index, 'X_centroid'] = x_centroid
                    top_ks_results.at[index, 'Y_centroid'] = y_centroid

            if save_dir is not None:
                top_ks_results.to_csv(os.path.join(save_dir, ks_sample_name + '_top_ks_results.csv'))
            ks_rank_grid_dict[ks_sample_name] = top_ks_results
    return ks_rank_grid_dict

In [24]:
ks_results_dir = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma/'
grid_files = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'
save_dir = '/michorlab/ecdyer/multiplex_spatial/tcm_results/cd45_sma_ranked/'

test_ks_rank_grid_dict = rank_and_calculate_distance(ks_results_dir, 
                                                     grid_files, 
                                                     save_dir=save_dir)

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class